<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day21_practice3_rag_service_%EC%99%84%EC%84%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 서버의 rag_service 완성
# LangChain 스택: Document → 청킹(500/100) → Chroma → similarity_search

In [2]:
!pip install langchain-core langchain-chroma langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [3]:
import os, re, shutil, zlib

USE_REAL_AI = False
FALLBACK_ON_ERROR = True

KNOWLEDGE_DIR = "knowledge" # 문서 폴더
PERSIST_DIR = "/tmp/chroma_rehearsal" # 임베딩 벡터 변환 결과, 원문 텍스트, 메타데이터가 지정 폴더 안에 실제 파일 형태로 저장, 이후에 로드해서 검색 가능
# 문서 내용 → 청킹되어 벡터 DB(chroma_db/)에 적재, 문서를 갱신했다면 chroma_db/를 지워 색인을 재구축

_vectorstore = None

def _reset_index(): #Chroma 색인 리셋
  global _vectorstore
  _vectorstore = None
  try:
    try:
      from chromadb.api.shared_system_client import SharedSystemClient
    except ImportError:
      from chromadb.api.client import SharedSystemClient
    SharedSystemClient.clear_system_cache()
  except Exception as e:
    print(f"Chroma 클라이먼트 캐시 정리 실패(무시): {type(e).__name__}: {e}")
  shutil.rmtree(PERSIST_DIR, ignore_errors=True) # 디스크 색인 삭제

_reset_index() # 노트북을 처음부터 다시 실행해도 항상 깨끗한 상태에서 시작

In [4]:
# 셀 1. knowledge/ 문서 준비
os.makedirs(KNOWLEDGE_DIR, exist_ok=True)
_docs = {
    "01_도로법_포트홀.md": "도로법 시행령 제55조에 따라 도로 관리청은 포트홀 등 "
        "도로 파손 발견 시 24시간 이내 응급 복구 조치를 시행해야 한다. "
        "심각도 4 이상은 긴급 보수팀을 우선 배정한다.",
    "02_맨홀_지침.md": "맨홀 관리 지침에 따라 맨홀 뚜껑 파손·이탈 시 즉시 관할 "
        "지자체에 신고하고 안전 펜스를 설치하여 차량과 보행자의 접근을 차단한다.",
    "03_균열_기준.md": "도로 유지관리 매뉴얼에 따라 균열 폭 3mm 이상은 보수 대상이다. "
        "방치 시 포트홀로 확대될 수 있으므로 주간 정기 점검에 포함한다.",
    "04_신고_절차.md": "안전신문고 앱으로 위치와 사진을 첨부해 신고한다. "
        "접수 후 관할 지자체가 처리하며 완료 시 신고자에게 통보한다.",
    "05_종합_도로관리_규정.md": (
        "제1조(긴급 보수) 심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다. "
        "배정 결과는 신고자에게 앱 알림으로 통보하며, 보수 완료 시 현장 사진을 첨부하여 처리 내역을 기록한다. "
        "긴급 보수가 지연될 경우 관할 부서장에게 자동으로 보고되며 사유를 명시해야 한다. "
        "제2조(정기 처리) 심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다. "
        "균열 폭이 3밀리미터 이상으로 확대되면 심각도를 재평가하여 우선순위를 조정한다. "
        "제3조(맨홀 관리) 맨홀 뚜껑 파손이나 이탈 신고는 안전 펜스를 우선 설치하여 2차 사고를 예방하고 "
        "상수도사업소에 이관한다. 이관 후에도 처리 현황을 추적하여 신고자에게 진행 상황을 안내한다. "
        "제4조(낙하물 대응) 도로 위 낙하물 신고는 2차 사고 예방을 위해 즉시 현장 출동을 원칙으로 하며 "
        "출동 전까지 해당 구간에 대한 서행 안내를 전광판에 표시한다. "
        "제5조(야간 접수) 야간 22시부터 06시 사이 접수된 신고는 익일 오전 담당자 배정 후 처리를 시작한다. "
        "다만 심각도 5의 위험은 야간에도 즉시 출동하여 긴급 조치한다. "
        "제6조(완료 통보) 모든 처리 완료 신고는 처리 전후 사진과 함께 신고자에게 앱 알림으로 통보한다."
    ),
}
for fname, text in _docs.items():
  with open(os.path.join(KNOWLEDGE_DIR, fname), "w", encoding="utf-8") as f:
    f.write(text)
print(f"knowledge/ 문서 {len(_docs)}개 준비 (실전: 팀이 규정 문서를 계속 추가)")

knowledge/ 문서 5개 준비 (실전: 팀이 규정 문서를 계속 추가)


In [5]:
# 셀 2. 임베딩 - LangChain으로 만든다
from langchain_core.embeddings import Embeddings
import os, requests

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

# API 호출 - openAI 호환 api 형식으로 코드 작성
EMBED_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
EMBED_MODEL = "gemini-embedding-001" # Gemini 임베딩 모델

class GeminiEmbeddings(Embeddings):
  def __init__(self, api_key):
    self.api_key = api_key
  def _embed_batch(self, texts):
    r = requests.post(
        f"{EMBED_BASE_URL}/embeddings",
        headers={"Authorization": f"Bearer {self.api_key}"},
        json={"model": EMBED_MODEL, "input": texts},
        timeout=30,
    )
    r.raise_for_status() # 상태코드가 4xx, 5xx면 HTTPError 예외를 발생
    return [item["embedding"] for item in r.json()["data"]]

  def embed_documents(self, texts): # 여러 문서 → 벡터들(Chroma 색인용)
    return self._embed_batch(texts) # list[list[float]]

  def embed_query(self, text): # 질문 1개 → 벡터(검색용)
    return self._embed_batch([text])[0] # list[float]

class NgramEmbeddings(Embeddings):
  def __init__(self, dim=512, n=2):
    self.dim, self.n = dim, n
  def _embed(self, text):
    v = [0.0] * self.dim # 0벡터
    t = re.sub(r"\s+", "", text)
    for i in range(len(t) - self.n + 1):
      v[zlib.crc32(t[i:i+self.n].encode("utf-8")) % self.dim] += 1.0 # 사전 없이 조각을 0-dim 위치로 매핑(해싱 트릭)
    s = sum(x*x for x in v) ** 0.5 or 1.0
    return [x / s for x in v]
  def embed_documents(self, texts): return [self._embed(t) for t in texts]
  def embed_query(self, text): return self._embed(text)

# 4) 임베더를 고르는 팩토리 함수
def get_embeddings():
  key = _get_api_key()
  if key:
    return GeminiEmbeddings(key)
  return NgramEmbeddings()

In [17]:
# 셀 3. 벡터 DB 구축

# 1) 문서 읽기: 폴더의 .txt/.md 파일 → Document 리스트
def _load_documents():
  from langchain_core.documents import Document # 본문(page_content) + 문서 정보(metadata)를 한 묶음으로 담는
  docs = []
  if not os.path.exists(KNOWLEDGE_DIR):
    print(f"[RAG] '{KNOWLEDGE_DIR}/' 폴더가 없습니다.")
    return docs
  for fname in sorted(os.listdir(KNOWLEDGE_DIR)):
    if fname.endswith(('.txt', '.md')):
      with open(os.path.join(KNOWLEDGE_DIR, fname), encoding="utf-8") as f:
        text = f.read().strip() # 파일 전체 내용
      if text:
        docs.append(Document(page_content=text, metadata={"source":fname}))
  return docs # list[Document]

# 2) 벡터 DB 만들기: 청킹 → 임베딩 → Chroma 저장
def _build_vectorstore(embeddings):
  from langchain_chroma import Chroma # 벡터를 저장, 검색하고 디스크에 문서 원문 저장까지 해 주는 벡터DB
  from langchain_text_splitters import RecursiveCharacterTextSplitter # 긴 문서를 일정 길이 조각으로 나누는 도구

  os.makedirs(PERSIST_DIR, exist_ok=True)
  if os.listdir(PERSIST_DIR): # 색인이 이미 있으면 재적재 없이 로드
    store = Chroma(collection_name="road_regulations", embedding_function=embeddings, persist_directory=PERSIST_DIR)
    if store.get(limit=1)["ids"]: # 문서 조각이 실제로 들어 있을 때만 재사용
      print("[RAG] 기존 색인을 로드합니다 (재 적재 없음)")
      return store # 아래 청킹, 임베딩 코드는 실행하지 않고 함수 종료

  # 신규 생성
  docs = _load_documents()
  if not docs:
    raise RuntimeError(
        f"'{KNOWLEDGE_DIR}/'에 규정 문서(.txt/.md)가 없습니다."
        "문서를 넣거나 USE_REAL_AI를 False로 두세요."
    )
  print(f"[RAG]문서 {len(docs)}개로 벡터 DB를 만듭니다...")
  splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100) # 조각 하나의 최대 길이(글자 수 기준), 인접한 조각이 100글자씩 겹치게 함
  # 문단 (\n\n) → 줄(\n) → 공백 → 글자 순으로 큰 구분자부터 시도
  chunks = splitter.split_documents(docs) # 청킹
  return Chroma.from_documents(documents=chunks, embedding=embeddings, collection_name="road_regulations", persist_directory=PERSIST_DIR)
  # 조각 본문을 embeddings.embed_documents로 벡터화 → 벡터·원문·metadata를 Chroma 벡터DB에 저장 → persist_directory 폴더에 디스크 저장

# 3) 벡터DB를 한 번만 만들어 재사용
def _get_vectorstore(): # 서버에서 이 함수로 벡터DB를 받게 하면 첫 호출 때만 만들고 이후는 캐시를 재사용함
  global _vectorstore
  if _vectorstore is None:
    _vectorstore = _build_vectorstore(get_embeddings()) # Chroma 벡터DB 객체를 전역 변수에 저장
  return _vectorstore

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

_preview_docs = [Document(page_content=t, metadata={"source": n}) for n, t in _docs.items()]
_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
_chunks = _splitter.split_documents(_preview_docs)
print(f"📄청킹 결과: 문서 {len(_preview_docs)}개 → 청크 {len(_chunks)}개")
for _c in _chunks:
  print(f"  [{_c.metadata["source"]}] {len(_c.page_content)}자")
_long = [c for c in _chunks if c.metadata["source"] == "05_종합_도로관리_규정.md"]
if len(_long) >= 2:
  print(f"\n첫 조각의 마지막 100자 :", _long[0].page_content[-100:])
  print("둘째 조각의 처음 100자 :", _long[1].page_content[:100])

📄청킹 결과: 문서 5개 → 청크 6개
  [01_도로법_포트홀.md] 95자
  [02_맨홀_지침.md] 73자
  [03_균열_기준.md] 74자
  [04_신고_절차.md] 62자
  [05_종합_도로관리_규정.md] 498자
  [05_종합_도로관리_규정.md] 211자

첫 조각의 마지막 100자 : 사고 예방을 위해 즉시 현장 출동을 원칙으로 하며 출동 전까지 해당 구간에 대한 서행 안내를 전광판에 표시한다. 제5조(야간 접수) 야간 22시부터 06시 사이 접수된 신고는 익일
둘째 조각의 처음 100자 : 예방을 위해 즉시 현장 출동을 원칙으로 하며 출동 전까지 해당 구간에 대한 서행 안내를 전광판에 표시한다. 제5조(야간 접수) 야간 22시부터 06시 사이 접수된 신고는 익일 오전


In [15]:
# 셀 4. 서비스 함수 : 서버

# 1) 출력 정리
def _normalize(regulations): # 서버가 받는 모양 확인
  out = []
  for r in regulations or []:
    if isinstance(r, dict):
      out.append({"title": str(r.get("title") or "관련 규정"), "source": str(r.get("source") or ""), "content": str(r.get("content") or "")})
  return out

# 2) 가짜(mock) 검색:
_MOCK_DOCS = [
    {"title": "도로법 시행령 제55조", "source": "도로법",
     "content": "포트홀 발견 시 24시간 이내 응급 복구 조치를 시행해야 한다."},
    {"title": "도로 파손 신고 절차", "source": "안전신문고",
     "content": "앱으로 위치·사진을 첨부해 신고하면 관할 지자체가 처리한다."},
]
def _mock_search_by_text(query, k=3):
  text = (query or "").replace(" ", "")
  scored = []
  for doc in _MOCK_DOCS:
    hay = (doc["title"] + doc["content"]).replace(" ", "")
    score = sum(1 for i in range(len(text) - 1) if text[i:i+2] in hay)
    scored.append((score, doc))
  scored.sort(key=lambda x: -x[0])
  return[d for _, d in scored[:k]]

# 3) 진짜 검색: Chroma 벡터DB
def _real_search_by_text(query, k=3):
  results = _get_vectorstore().similarity_search(query, k=k)
  return [{"title": doc.metadata.get("source", "관련 규정"),"source": doc.metadata.get("source", "지식베이스"),
           "content": doc.page_content[:150] + ("..." if len(doc.page_content) > 150 else "")} for doc in results]

# 4) 입구 함수
def search_by_text(query, k=3):
  if not USE_REAL_AI:
    return _normalize(_mock_search_by_text(query, k))
  try:
    return _normalize(_real_search_by_text(query, k))
  except Exception as e:
    print(f"[RAG] 검색 실패: {type(e).__name__}: {e}")
    if not FALLBACK_ON_ERROR:
      raise # 지금 처리 중인 예외를 그대로 다시 던짐, mock으로 조용히 대체 금지
    print("[RAG] 가짜 검색으로 대체합니다")
    return _normalize(_mock_search_by_text(query, k))

In [21]:
# 셀 5. 검증

# 1) real 경로 스위치 켜기
USE_REAL_AI = True
FALLBACK_ON_ERROR = False

# 2) 테스트 질문 준비
질문들 = ["포트홀은 얼마나 빨리 복구해야 하나요?",
          "맨홀 뚜껑이 빠졌을 때 어떻게 하나요?",
          "신고는 어디로 하나요?"]
for q in 질문들:
  results = search_by_text(q, k=2)
  assert isinstance(results, list) and len(results) >= 1
  for r in results:
    assert set(r.keys()) == {"title", "source", "content"}
    assert all(isinstance(v, str) for v in r.values()) # 값이 전부 문자열이어야 서버, 화면에서 타입 에러가 나지 않음
  print(f"\n Q : {q}")
  print(f"1위: [{results[0]["source"]}] {results[0]["content"]}...")
print("\nreal 경로 계약 검증 통과 (title/source/content 전부 str)")


 Q : 포트홀은 얼마나 빨리 복구해야 하나요?
1위: [01_도로법_포트홀.md] 도로법 시행령 제55조에 따라 도로 관리청은 포트홀 등 도로 파손 발견 시 24시간 이내 응급 복구 조치를 시행해야 한다. 심각도 4 이상은 긴급 보수팀을 우선 배정한다....

 Q : 맨홀 뚜껑이 빠졌을 때 어떻게 하나요?
1위: [02_맨홀_지침.md] 맨홀 관리 지침에 따라 맨홀 뚜껑 파손·이탈 시 즉시 관할 지자체에 신고하고 안전 펜스를 설치하여 차량과 보행자의 접근을 차단한다....

 Q : 신고는 어디로 하나요?
1위: [04_신고_절차.md] 안전신문고 앱으로 위치와 사진을 첨부해 신고한다. 접수 후 관할 지자체가 처리하며 완료 시 신고자에게 통보한다....

real 경로 계약 검증 통과 (title/source/content 전부 str)
